# 03 — Sector Analysis
Oil sensitivity (lag 0/1/2), regime-split correlations.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from event_study import oil_sensitivity, get_sector_returns, OIL_SHOCK_EVENTS
from utils import DATA_PROC, TABLES_DIR, PLOTS_DIR, set_theme, save_table
from constants import MARKET_COL, OIL_COL, FX_COL, SECTORS


In [2]:
returns_raw = pd.read_parquet(DATA_PROC / 'returns.parquet')
returns     = get_sector_returns(returns_raw)
print(f'Sector cols: {[c for c in returns.columns if c not in (MARKET_COL, OIL_COL, FX_COL)]}')


Sector cols: ['OIL_GAS', 'AUTO', 'FMCG', 'IT', 'PHARMA']


In [3]:
# Oil sensitivity at lags 0, 1, 2
lag_results = {}
for lag in [0, 1, 2]:
    df = oil_sensitivity(returns_raw, lag=lag)
    lag_results[lag] = df
    save_table(df.reset_index(), f'oil_sensitivity_lag{lag}')

print('Lag-0 (contemporaneous):')
print(lag_results[0].to_string())


Saved table → C:\Users\anves\projects\osi_clean\outputs\tables\oil_sensitivity_lag0.csv
Saved table → C:\Users\anves\projects\osi_clean\outputs\tables\oil_sensitivity_lag1.csv
Saved table → C:\Users\anves\projects\osi_clean\outputs\tables\oil_sensitivity_lag2.csv
Lag-0 (contemporaneous):
         lag    corr  p_value  significant
sector                                    
OIL_GAS    0 -0.0108   0.6354        False
AUTO       0 -0.0052   0.8209        False
FMCG       0  0.0021   0.9270        False
IT         0  0.0069   0.7627        False
PHARMA     0  0.0040   0.8597        False


In [4]:
# Sector cumulative returns
set_theme()
brent = returns[OIL_COL]
fig, axes = plt.subplots(len(SECTORS), 1, figsize=(13, 3 * len(SECTORS)), sharex=True)
for ax, sec in zip(axes, SECTORS):
    ax.plot(returns.index, returns[sec].cumsum() * 100, lw=1.5, label=sec)
    ax.set_ylabel(f'{sec}\nCum. ret %')
    ax.legend(loc='upper left', fontsize=8)
axes[-1].set_xlabel('Date')
fig.suptitle('Sector cumulative returns (log)', y=1.01)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'sector_cumulative_returns.png', dpi=150, bbox_inches='tight')
plt.show()


In [5]:
# Regime-split oil sensitivity
nifty_vol       = returns[MARKET_COL].rolling(20).std() * (252 ** 0.5)
high_vol_regime = nifty_vol > 0.25
print(f'High-vol days: {high_vol_regime.sum()} ({high_vol_regime.mean()*100:.1f}%)')

print('\nOil-sector correlations by regime:')
regime_rows = []
for sec in SECTORS:
    normal   = returns.loc[~high_vol_regime].dropna()
    stressed = returns.loc[high_vol_regime].dropna()
    r_n, _ = stats.pearsonr(normal[OIL_COL],   normal[sec])
    r_s, _ = stats.pearsonr(stressed[OIL_COL], stressed[sec])
    print(f'  {sec:10s}  normal={r_n:+.3f}  stressed={r_s:+.3f}')
    regime_rows.append({'sector': sec, 'corr_normal': round(r_n,4), 'corr_stressed': round(r_s,4)})

save_table(pd.DataFrame(regime_rows), 'regime_car_comparison')
print('NB03 complete ✓')


High-vol days: 48 (2.5%)

Oil-sector correlations by regime:
  OIL_GAS     normal=-0.010  stressed=-0.028
  AUTO        normal=-0.001  stressed=-0.023
  FMCG        normal=+0.008  stressed=-0.009
  IT          normal=+0.001  stressed=+0.016
  PHARMA      normal=-0.001  stressed=+0.046
Saved table → C:\Users\anves\projects\osi_clean\outputs\tables\regime_car_comparison.csv
NB03 complete ✓
